# 05 — Testing box2boxcam on a video

This notebook applies a trained model to **every frame of a video**, draws the ball detections, then creates a new MP4 video.

It never loads the whole video into memory: one frame is read, analyzed and written at a time. It can therefore be used on a video several minutes long without reproducing the memory problem encountered with large image galleries.

> Wait for training to finish before running this notebook. The `best.pt` file may still change during training.

In [ ]:
import shutil
import subprocess
import time
from pathlib import Path

import cv2
import torch
from IPython.display import Markdown, Video, display
from tqdm.auto import tqdm
from ultralytics import YOLO

# Works whether Jupyter was opened from the project root or from notebooks/.
PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (folder / "pyproject.toml").exists()
)
print(f"Project: {PROJECT_ROOT}")

## 1. Parameters

The two essential settings are:

- `VIDEO_PATH`: path to the video to analyze;
- `MODEL_PATH`: `None` automatically picks the most recent `best.pt`. A specific path can also be provided.

`CONFIDENCE = 0.25` means a prediction is drawn from 25% confidence. Raising this threshold reduces false alarms, but risks missing more balls.

In [ ]:
# TO EDIT: put the video wherever you like, then set its path here.
VIDEO_PATH = PROJECT_ROOT / "data/external/videos/my_video.mp4"

# None = latest best.pt produced in models/checkpoints/baseline_runs/.
MODEL_PATH = None

CONFIDENCE = 0.25
IMAGE_SIZE = 640
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
OUTPUT_MAX_WIDTH = 1920  # only downsizes the output video if the source is 4K
MAX_FRAMES = None        # e.g. 300 to test on the first 300 frames only

OUTPUT_DIR = PROJECT_ROOT / "models/reports/video_inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_PATH is None:
    candidates = sorted(
        (PROJECT_ROOT / "models/checkpoints/baseline_runs").glob("*/train/weights/best.pt"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError("No best.pt found. Finish a training run first.")
    MODEL_PATH = candidates[0]
else:
    MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()

VIDEO_PATH = Path(VIDEO_PATH).expanduser().resolve()
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

print(f"Video  : {VIDEO_PATH}")
print(f"Model  : {MODEL_PATH}")
print(f"Device : {DEVICE}")

## 2. Read the video properties

This step does not run the model yet. It lets you check the duration, the frame rate and the resolution.

In [ ]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f"OpenCV cannot open: {VIDEO_PATH}")

FPS = capture.get(cv2.CAP_PROP_FPS) or 30.0
FRAME_COUNT = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
SOURCE_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.release()

frames_to_process = min(FRAME_COUNT, MAX_FRAMES) if MAX_FRAMES else FRAME_COUNT
duration_seconds = frames_to_process / FPS
display(Markdown(
    f"**Source:** {SOURCE_WIDTH} × {SOURCE_HEIGHT} — {FPS:.2f} frames/s — "
    f"{frames_to_process:,} frames to process — {duration_seconds / 60:.1f} minute(s)."
))

## 3. Detect the ball and create the video

For each frame:

1. YOLO temporarily resizes the frame to `IMAGE_SIZE=640` for inference;
2. the predicted coordinates are mapped back to the original frame;
3. a red rectangle and the confidence are drawn;
4. the annotated frame is immediately written to the output video.

The original audio is then copied back with `ffmpeg` when it is available.

In [ ]:
model = YOLO(str(MODEL_PATH))

scale = min(1.0, OUTPUT_MAX_WIDTH / SOURCE_WIDTH)
output_width = int(SOURCE_WIDTH * scale)
output_height = int(SOURCE_HEIGHT * scale)
# Some codecs require even dimensions.
output_width -= output_width % 2
output_height -= output_height % 2

run_name = MODEL_PATH.parent.parent.parent.name
SILENT_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}__{run_name}__silent.mp4"
FINAL_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}__{run_name}__ball.mp4"

capture = cv2.VideoCapture(str(VIDEO_PATH))
writer = cv2.VideoWriter(
    str(SILENT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (output_width, output_height),
)
if not writer.isOpened():
    capture.release()
    raise RuntimeError("Unable to create the temporary video with the mp4v codec.")

started_at = time.perf_counter()
processed = 0
frames_with_detection = 0
detections = 0

try:
    with tqdm(total=frames_to_process, unit="frame", desc="Video inference") as progress:
        while processed < frames_to_process:
            ok, frame = capture.read()
            if not ok:
                break

            result = model.predict(
                frame,
                imgsz=IMAGE_SIZE,
                conf=CONFIDENCE,
                device=DEVICE,
                verbose=False,
            )[0]

            boxes = result.boxes
            if boxes is not None and len(boxes):
                frames_with_detection += 1
                detections += len(boxes)
                for xyxy, confidence in zip(boxes.xyxy.cpu().numpy(), boxes.conf.cpu().numpy()):
                    x1, y1, x2, y2 = (int(value) for value in xyxy)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                    label = f"ball {confidence:.2f}"
                    text_y = max(22, y1 - 8)
                    cv2.putText(
                        frame, label, (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, (0, 0, 255), 2, cv2.LINE_AA,
                    )

            if (output_width, output_height) != (SOURCE_WIDTH, SOURCE_HEIGHT):
                frame = cv2.resize(frame, (output_width, output_height), interpolation=cv2.INTER_AREA)
            writer.write(frame)
            processed += 1
            progress.update(1)
finally:
    capture.release()
    writer.release()

# Re-encode to H.264, playable in most players, and recover the audio if any.
ffmpeg = shutil.which("ffmpeg")
if ffmpeg:
    subprocess.run(
        [
            ffmpeg, "-y", "-loglevel", "error",
            "-i", str(SILENT_VIDEO), "-i", str(VIDEO_PATH),
            "-map", "0:v:0", "-map", "1:a?",
            "-c:v", "libx264", "-preset", "fast", "-crf", "20",
            "-c:a", "aac", "-shortest", str(FINAL_VIDEO),
        ],
        check=True,
    )
    SILENT_VIDEO.unlink()
else:
    if FINAL_VIDEO.exists():
        FINAL_VIDEO.unlink()
    SILENT_VIDEO.rename(FINAL_VIDEO)

elapsed = time.perf_counter() - started_at
display(Markdown(
    f"### Done\n"
    f"- **Time:** {elapsed / 60:.1f} minute(s)\n"
    f"- **Average speed:** {processed / elapsed:.1f} frames/s\n"
    f"- **Frames with at least one detection:** {frames_with_detection:,} / {processed:,}\n"
    f"- **Total number of boxes drawn:** {detections:,}\n"
    f"- **File:** `{FINAL_VIDEO}`"
))

## 4. Look at the result

The video stays stored as a separate file: it is not embedded in the notebook, so the `.ipynb` file does not become huge.

In [ ]:
display(Video(filename=str(FINAL_VIDEO), embed=False, width=960))